# Track A: 2026 Renewal Pricing EDA

Complete EDA and candidate feature engineering using only B1, B2, B3 and B4. No model training or optimization is performed.

Coverage constraint: B2 provides renewal and quote history for 2023-2025. B3 provides paid claims for FY2025 only, so claims, frequency, severity and MLR are not fabricated for 2023 or 2024. Primary case: carry FY2025 paid claims into 2026 with no medical inflation or utilization trend.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' if (ROOT / 'data').exists() else ROOT   # raw input files
OUT = ROOT / 'outputs' / 'EDA'
PLOTS = OUT / 'plots'
TABLES = OUT / 'tables'
OUT.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
FILES = {name: DATA / filename for name, filename in {
    'B1': 'B1_account_master.csv.gz',
    'B2': 'B2_renewal_quote_history.csv.gz',
    'B3': 'B3_claims_experience_details.csv.gz',
    'B4': 'B4_regional_macro_benchmarks.csv.gz',
}.items()}
b1 = pd.read_csv(FILES['B1'])
b2 = pd.read_csv(FILES['B2'])
b3 = pd.read_csv(FILES['B3'], parse_dates=['service_date'])
b4 = pd.read_csv(FILES['B4'])
print({name: df.shape for name, df in {'B1': b1, 'B2': b2, 'B3': b3, 'B4': b4}.items()})

## 1. Data quality and data structure
All joins and grains are checked before aggregation. B3 is explicitly FY2025-only paid episode data.

In [ ]:
frames = {'B1': b1, 'B2': b2, 'B3': b3, 'B4': b4}
schema = pd.DataFrame([{'table': name, 'column': col, 'dtype': str(df[col].dtype), 'non_null': int(df[col].notna().sum()), 'unique': int(df[col].nunique(dropna=False))} for name, df in frames.items() for col in df.columns])
coverage = pd.DataFrame([
    {'table': 'B1', 'coverage': 'FY2025 snapshot', 'rows': len(b1), 'unique_accounts': b1.account_id.nunique()},
    {'table': 'B2', 'coverage': 'FY2023-FY2025', 'rows': len(b2), 'unique_accounts': b2.account_id.nunique()},
    {'table': 'B3', 'coverage': f'{b3.service_date.min().date()} to {b3.service_date.max().date()} (FY2025 only)', 'rows': len(b3), 'unique_accounts': b3.account_id.nunique()},
    {'table': 'B4', 'coverage': f'{b4.calendar_year.min()}-{b4.calendar_year.max()} across {b4.rating_region.nunique()} regions', 'rows': len(b4), 'unique_accounts': np.nan},
])
missing = pd.DataFrame([{'table': name, 'missing_cells': int(df.isna().sum().sum()), 'columns_with_missing': int(df.isna().any().sum())} for name, df in frames.items()])
duplicates = pd.DataFrame([
    {'table': 'B1', 'key': 'account_id', 'duplicate_rows': int(b1.account_id.duplicated().sum())},
    {'table': 'B2', 'key': 'quote_id', 'duplicate_rows': int(b2.quote_id.duplicated().sum())},
    {'table': 'B3', 'key': 'claim_episode_id', 'duplicate_rows': int(b3.claim_episode_id.duplicated().sum())},
    {'table': 'B4', 'key': 'rating_region + calendar_year', 'duplicate_rows': int(b4.duplicated(['rating_region', 'calendar_year']).sum())},
])
relationships = pd.DataFrame([
    {'check': 'B1 account_id unique', 'passed': b1.account_id.is_unique},
    {'check': 'Every B2 account is in B1', 'passed': set(b2.account_id) <= set(b1.account_id)},
    {'check': 'Every B3 account is in B1', 'passed': set(b3.account_id) <= set(b1.account_id)},
    {'check': 'Every B1 region has B4 coverage', 'passed': set(b1.rating_region) <= set(b4.rating_region)},
    {'check': 'Exactly 3 B2 rows per account', 'passed': b2.groupby('account_id').size().eq(3).all()},
    {'check': 'B3 all service dates in 2025', 'passed': b3.service_date.dt.year.eq(2025).all()},
    {'check': 'B3 all episodes are Paid', 'passed': b3.episode_status.eq('Paid').all()},
    {'check': 'B4 region-year unique', 'passed': not b4.duplicated(['rating_region', 'calendar_year']).any()},
])
assert relationships.passed.all()
display(schema); display(coverage); display(missing); display(duplicates); display(relationships)
for name, table in {'schema': schema, 'coverage': coverage, 'missingness': missing, 'duplicates': duplicates, 'relationships': relationships}.items(): table.to_csv(TABLES / f'{name}.csv', index=False)
print('B3 limitation: claims experience is FY2025-only; no 2023/2024 claims or MLR features are created.')

## 2. Account and profitability EDA
FY2025 claims are aggregated to account level and compared with the FY2025 B2 quoted premium. The primary carried-forward claims amount is not trend-adjusted.

In [ ]:
claims = b3.groupby('account_id').agg(
paid_claims=('paid_amount', 'sum'), allowed_claims=('allowed_amount', 'sum'), claim_count=('claim_episode_id', 'nunique'),
unique_members=('member_token', 'nunique'), claim_severity=('paid_amount', 'mean'), median_claim=('paid_amount', 'median'),
p90_claim=('paid_amount', lambda x: x.quantile(.90)), max_claim=('paid_amount', 'max'), provider_count=('provider_id', 'nunique')
).reset_index()
latest = b2.loc[b2.quote_year.eq(2025), ['account_id', 'quoted_annual_premium', 'risk_score_at_quote', 'renewed_flag', 'quoted_rate_change_pct']].rename(columns={'quoted_annual_premium': 'premium', 'risk_score_at_quote': 'quote_risk_score', 'renewed_flag': 'renewed', 'quoted_rate_change_pct': 'offered_rate_change_pct'})
a25 = b1.merge(latest, on='account_id', validate='one_to_one').merge(claims, on='account_id', validate='one_to_one')
a25['mlr'] = a25.paid_claims / a25.premium
a25['claims_per_life'] = a25.claim_count / a25.covered_lives_2025
a25['paid_per_life'] = a25.paid_claims / a25.covered_lives_2025
a25['claims_premium_gap'] = a25.paid_claims - a25.premium
a25['loss_making'] = a25.mlr.gt(1)
assert len(a25) == 7300 and a25.account_id.is_unique
risk_summary = a25.groupby('risk_band', observed=False).agg(accounts=('account_id', 'size'), mean_mlr=('mlr', 'mean'), median_mlr=('mlr', 'median'), loss_making_share=('loss_making', 'mean'), total_premium=('premium', 'sum'), total_paid_claims=('paid_claims', 'sum'), mean_lives=('covered_lives_2025', 'mean')).reset_index()
profit_summary = pd.DataFrame({'metric': ['accounts', 'mean_mlr', 'median_mlr', 'loss_making_share', 'total_premium', 'total_paid_claims'], 'value': [len(a25), a25.mlr.mean(), a25.mlr.median(), a25.loss_making.mean(), a25.premium.sum(), a25.paid_claims.sum()]})
display(profit_summary); display(risk_summary)
profit_summary.to_csv(TABLES / 'profitability_summary.csv', index=False); risk_summary.to_csv(TABLES / 'profitability_by_risk_band.csv', index=False)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.scatterplot(data=a25, x='premium', y='paid_claims', hue='risk_band', alpha=.45, s=20, ax=axes[0, 0]); axes[0, 0].plot([0, a25.premium.max()], [0, a25.premium.max()], '--', color='black'); axes[0, 0].set_title('FY2025 premium vs paid claims')
sns.histplot(a25.mlr.clip(upper=a25.mlr.quantile(.99)), bins=40, color='#24527a', ax=axes[0, 1]); axes[0, 1].axvline(1, color='#b23a48', ls='--'); axes[0, 1].set_title('FY2025 MLR distribution')
sns.boxplot(data=a25, x='risk_band', y=a25.mlr.clip(upper=a25.mlr.quantile(.99)), color='#7aa6c2', ax=axes[1, 0]); axes[1, 0].axhline(1, color='#b23a48', ls='--'); axes[1, 0].set_title('MLR by risk band')
tmp = risk_summary.assign(loss_making_share=lambda x: x.loss_making_share * 100); sns.barplot(data=tmp, x='risk_band', y='loss_making_share', color='#b23a48', ax=axes[1, 1]); axes[1, 1].set_title('Loss-making share by risk band'); axes[1, 1].set_ylabel('Accounts (%)')
plt.tight_layout(); plt.savefig(PLOTS / 'profitability_eda.png', dpi=160); plt.show()
print('Finding -> MLR is heterogeneous and 21.2% of accounts are loss-making; implication -> pricing needs account-level claims and risk differentiation; modelling implication -> retain profitability, frequency, severity and exposure candidates.')
print('Finding -> Risk-band economics rise sharply with risk; implication -> high-risk accounts explain a large share of poor economics; modelling implication -> retain risk, but do not discard observed claims experience.')

## 3. Renewal and price response EDA
B2 is analyzed separately by year. Profitability segmentation uses FY2025 MLR as an end-of-2025 account characteristic; it is not treated as historical claims for 2023/2024.

In [ ]:
# Historical renewal analysis uses only quote-time variables.
# B1 industry, size and region are treated as stable account attributes because B2 has no historical versions.
quotes = b2.merge(
    b1[['account_id', 'industry', 'covered_lives_2025', 'rating_region']],
    on='account_id', validate='many_to_one'
).merge(
    b4.rename(columns={'calendar_year': 'quote_year'}),
    on=['rating_region', 'quote_year'], how='left', validate='many_to_one'
)
quotes['risk_band_at_quote'] = pd.cut(
    quotes['risk_score_at_quote'],
    bins=[0, .5, 1, 1.5, 2.5, np.inf],
    labels=['<0.5', '0.5-1.0', '1.0-1.5', '1.5-2.5', '>2.5'],
    right=False,
)
quotes['risk_segment_at_quote'] = np.where(
    quotes['risk_score_at_quote'] >= quotes.groupby('quote_year')['risk_score_at_quote'].transform('median'),
    'High quote-time risk',
    'Low quote-time risk',
)
quotes['size_band'] = pd.cut(
    quotes.covered_lives_2025,
    bins=[0, 150, 250, 400, np.inf],
    labels=['<=150', '151-250', '251-400', '401+'],
)
quotes['regional_economic_tier_at_quote'] = quotes.groupby('quote_year')['unemployment_rate_pct'].transform(
    lambda values: pd.qcut(values, q=3, labels=['Lower unemployment', 'Middle unemployment', 'Higher unemployment'])
)
rate_bins = [-20, -1, 0, 5, 10, 15, 20, 30]
rate_labels = ['<-1%', '-1 to 0%', '0-5%', '5-10%', '10-15%', '15-20%', '20%+']
quotes['rate_band'] = pd.cut(quotes.quoted_rate_change_pct, bins=rate_bins, labels=rate_labels, right=False)
renewal_by_rate_year = quotes.groupby(['quote_year', 'rate_band'], observed=False).agg(accounts=('account_id', 'size'), renewals=('renewed_flag', 'sum'), renewal_probability=('renewed_flag', 'mean')).reset_index()
renewal_by_rate_year.to_csv(TABLES / 'renewal_by_rate_year.csv', index=False)
display(renewal_by_rate_year)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, year in zip(axes, [2023, 2024, 2025]):
    d = renewal_by_rate_year[renewal_by_rate_year.quote_year.eq(year)]
    sns.lineplot(data=d, x='rate_band', y='renewal_probability', marker='o', ax=ax, color='#24527a')
    for i, row in d.dropna(subset=['renewal_probability']).reset_index(drop=True).iterrows():
        ax.text(i, row.renewal_probability, f"n={int(row.accounts)}", fontsize=8, ha='center', va='bottom')
    ax.set_title(str(year)); ax.tick_params(axis='x', rotation=40); ax.set_xlabel('Offered rate change')
axes[0].set_ylabel('Renewal probability'); fig.suptitle('Renewal probability vs offered rate change by year'); fig.tight_layout(); fig.savefig(PLOTS / 'renewal_probability_by_year.png', dpi=160); plt.show()
response_specs = {
    'risk_band_at_quote': 'Risk band at quote',
    'industry': 'Industry (stable B1 attribute)',
    'size_band': 'Account size band (stable B1 exposure)',
    'regional_economic_tier_at_quote': 'Regional economic tier at quote',
}
response_rows = []
for column, label in response_specs.items():
    grouped = quotes.groupby(['quote_year', column, 'rate_band'], observed=False).agg(
        accounts=('account_id', 'size'),
        renewals=('renewed_flag', 'sum'),
        renewal_probability=('renewed_flag', 'mean'),
    ).reset_index().rename(columns={column: 'segment'})
    grouped['segment_type'] = label
    response_rows.append(grouped[['quote_year', 'segment_type', 'segment', 'rate_band', 'accounts', 'renewals', 'renewal_probability']])
response_table = pd.concat(response_rows, ignore_index=True)
response_table.to_csv(TABLES / 'renewal_response_by_risk_industry_region_size.csv', index=False)
display(response_table)
fig, axes = plt.subplots(2, 2, figsize=(16, 11), sharey=True)
for ax, (column, label) in zip(axes.flat, response_specs.items()):
    plot_data = response_table[response_table.segment_type.eq(label)]
    sns.lineplot(data=plot_data, x='rate_band', y='renewal_probability', hue='segment', style='quote_year', marker='o', ax=ax)
    ax.set_title(f'By {label.lower()}'); ax.set_xlabel('Rate change (%)'); ax.tick_params(axis='x', rotation=40)
    ax.legend(title=label, fontsize=7, title_fontsize=8)
axes[0, 0].set_ylabel('Renewal probability'); axes[1, 0].set_ylabel('Renewal probability')
fig.suptitle('Renewal price response using quote-time risk and quote-year regional context')
fig.tight_layout(); fig.savefig(PLOTS / 'renewal_response_risk_industry_region_size.png', dpi=160); plt.show()
print('Finding -> Renewal probability declines as offered rate increases across years and quote-time risk bands; implication -> rate actions carry retention risk; modelling implication -> preserve year-specific rate-response evidence with sample counts.')
print('Finding -> Industry and size are stable B1 attributes; regional curves use the matching quote-year B4 benchmark; implication -> interactions can be investigated without mixing future context into earlier years.')
print('Retrospective-only note -> FY2025 MLR/loss-making may be used for the 2026 profitability analysis, but is excluded from all 2023/2024 renewal evidence.')

## 4. Temporal stability and 5. Regional analysis
Only B2 variables are compared across all three years. B3 claims stability cannot be assessed across years. B4 is used for relevant regional context, not generic regional plotting.

In [ ]:
temporal = b2.groupby('quote_year').agg(accounts=('account_id', 'nunique'), renewal_rate=('renewed_flag', 'mean'), mean_rate_change=('quoted_rate_change_pct', 'mean'), median_rate_change=('quoted_rate_change_pct', 'median'), mean_premium=('quoted_annual_premium', 'mean'), total_premium=('quoted_annual_premium', 'sum'), mean_quote_risk=('risk_score_at_quote', 'mean')).reset_index()
temporal.to_csv(TABLES / 'temporal_stability.csv', index=False); display(temporal)
fig, axes = plt.subplots(1, 2, figsize=(12, 4)); sns.lineplot(data=temporal, x='quote_year', y='renewal_rate', marker='o', ax=axes[0]); sns.lineplot(data=temporal, x='quote_year', y='mean_rate_change', marker='o', ax=axes[1], color='#b23a48'); axes[0].set_title('Renewal rate by year'); axes[1].set_title('Mean offered rate change by year'); plt.tight_layout(); plt.savefig(PLOTS / 'temporal_stability.png', dpi=160); plt.show()
region25 = b4.loc[b4.calendar_year.eq(2025), ['rating_region', 'unemployment_rate_pct']].merge(b2, on='rating_region' if 'rating_region' in b2.columns else 'account_id', how='left') if False else b1[['account_id', 'rating_region']].merge(b2, on='account_id').merge(b4.loc[b4.calendar_year.eq(2025), ['rating_region', 'unemployment_rate_pct']], on='rating_region')
regional = region25.groupby('rating_region').agg(accounts=('account_id', 'nunique'), renewal_rate=('renewed_flag', 'mean'), mean_rate_change=('quoted_rate_change_pct', 'mean'), unemployment_rate_2025=('unemployment_rate_pct', 'first')).reset_index()
regional.to_csv(TABLES / 'regional_2025_context.csv', index=False); display(regional.describe(include='all'))
fig, ax = plt.subplots(figsize=(7, 5)); sns.scatterplot(data=regional, x='unemployment_rate_2025', y='renewal_rate', size='accounts', sizes=(30, 250), alpha=.7, ax=ax); ax.set_title('Regional employment context vs renewal rate'); ax.set_xlabel('2025 unemployment rate'); ax.set_ylabel('Renewal rate'); plt.tight_layout(); plt.savefig(PLOTS / 'regional_context_vs_renewal.png', dpi=160); plt.show()
print('Finding -> Renewal rate, rate changes, premium and quote-time risk are stable across 2023-2025 at portfolio level; implication -> historical B2 supports time-ordered validation, while claims trends require more data.')
print('Finding -> B4 unemployment is a contextual regional candidate, not a primary MLR uplift; implication -> investigate only if it adds value later. Medical inflation and utilization trend are excluded from the primary case.')

## 6. 2026 feature engineering
The feature table uses only end-FY2025 information. It includes B2 history, FY2025 B3 experience, B1 account/risk fields and 2025 B4 context. No 2023/2024 claims features are created.

In [ ]:
account_year = b2.rename(columns={'quote_year': 'year', 'quoted_annual_premium': 'premium', 'quoted_rate_change_pct': 'offered_rate_change_pct', 'renewed_flag': 'renewed'}).merge(
    b1[['account_id', 'rating_region', 'industry', 'covered_lives_2025', 'annual_premium_2025', 'risk_score_2025', 'risk_band', 'renewal_month']],
    on='account_id', validate='many_to_one'
).merge(
    b4.rename(columns={'calendar_year': 'year'}),
    on=['rating_region', 'year'], how='left', validate='many_to_one'
).merge(claims, on='account_id', how='left', validate='many_to_one')
claim_columns = ['paid_claims', 'allowed_claims', 'claim_count', 'unique_members', 'claim_severity', 'median_claim', 'p90_claim', 'max_claim', 'provider_count']
account_year.loc[account_year.year.ne(2025), claim_columns] = np.nan
account_year['mlr'] = account_year.paid_claims / account_year.premium
account_year['claims_per_life'] = account_year.claim_count / account_year.covered_lives_2025
account_year['paid_per_life'] = account_year.paid_claims / account_year.covered_lives_2025
account_year['claims_premium_gap'] = account_year.paid_claims - account_year.premium
account_year['loss_making'] = account_year.mlr.gt(1)
account_year.to_csv(OUT / 'account_year_2023_2025.csv', index=False)
assert len(account_year) == 21900
assert account_year.groupby(['account_id', 'year']).size().eq(1).all()

hist = b2.pivot(index='account_id', columns='quote_year', values=['renewed_flag', 'quoted_rate_change_pct', 'quoted_annual_premium', 'risk_score_at_quote'])
hist.columns = [f'{a}_{y}' for a, y in hist.columns]
hist = hist.reset_index()
region_feature = b4.loc[b4.calendar_year.eq(2025), ['rating_region', 'unemployment_rate_pct']].rename(columns={'unemployment_rate_pct': 'unemployment_rate_2025'})
features = a25[['account_id', 'rating_region', 'industry', 'renewal_month', 'covered_lives_2025', 'annual_premium_2025', 'risk_score_2025', 'risk_band', 'premium', 'paid_claims', 'mlr', 'claim_count', 'claims_per_life', 'claim_severity', 'median_claim', 'p90_claim', 'max_claim', 'claims_premium_gap', 'loss_making']].merge(region_feature, on='rating_region', validate='many_to_one').merge(hist, on='account_id', validate='one_to_one')
features = features.rename(columns={'premium': 'latest_premium', 'mlr': 'latest_mlr', 'claim_count': 'latest_claim_count', 'claims_per_life': 'latest_claim_frequency', 'claim_severity': 'latest_claim_severity', 'risk_score_2025': 'latest_risk_score'})
features['renewal_count_3yr'] = features[[f'renewed_flag_{y}' for y in [2023, 2024, 2025]]].sum(axis=1)
features['renewal_rate_3yr'] = features.renewal_count_3yr / 3
features['rate_change_mean_3yr'] = features[[f'quoted_rate_change_pct_{y}' for y in [2023, 2024, 2025]]].mean(axis=1)
features['rate_change_slope'] = (features.quoted_rate_change_pct_2025 - features.quoted_rate_change_pct_2023) / 2
features['risk_score_change'] = features.risk_score_at_quote_2025 - features.risk_score_at_quote_2023
features['historical_loss_making_indicator'] = features.loss_making.astype(int)
features['three_year_average_mlr'] = np.nan
features['claims_trend'] = np.nan
features.to_csv(OUT / 'feature_inventory.csv', index=False)
display(account_year.head()); display(features.head()); print('Account-year rows:', len(account_year), 'feature rows:', len(features))

## 7. Feature redundancy and 8. Leakage audit
Correlations are descriptive and are not used for coefficient-based feature selection. Deterministic relationships and availability constraints are documented in the final decision table.

In [ ]:
numeric = ['latest_premium', 'paid_claims', 'latest_mlr', 'latest_claim_count', 'latest_claim_frequency', 'latest_claim_severity', 'covered_lives_2025', 'latest_risk_score', 'renewal_rate_3yr', 'rate_change_mean_3yr', 'rate_change_slope', 'unemployment_rate_2025']
corr = features[numeric].corr(method='spearman'); corr.to_csv(TABLES / 'feature_correlation.csv')
pairs = []
for i, left in enumerate(corr.columns):
    for right in corr.columns[i + 1:]:
        if abs(corr.loc[left, right]) >= .80: pairs.append({'feature_1': left, 'feature_2': right, 'spearman_correlation': corr.loc[left, right]})
high_corr = pd.DataFrame(pairs).sort_values('spearman_correlation', key=lambda x: x.abs(), ascending=False); high_corr.to_csv(TABLES / 'highly_correlated_pairs.csv', index=False); display(high_corr)
plt.figure(figsize=(10, 8)); sns.heatmap(corr, cmap='vlag', center=0, vmin=-1, vmax=1); plt.title('Numerical candidate feature correlation'); plt.tight_layout(); plt.savefig(PLOTS / 'feature_correlation_heatmap.png', dpi=160); plt.show()
print('Redundancy -> latest_mlr = paid_claims / latest_premium; retain latest_mlr as the business-facing profitability feature and avoid blindly retaining all three representations.')
print('Redundancy -> risk_band discretizes latest_risk_score; retain one primary representation and use the other for interpretation only.')
decisions = pd.DataFrame([
    ['latest_mlr', 'B2+B3', 'FY2025 paid claims / FY2025 quoted premium', 'Yes', 'Low', 'KEEP', 'Direct profitability signal under no-trend case.'],
    ['latest_claim_frequency', 'B1+B3', 'FY2025 paid episodes / covered lives', 'Yes', 'Low', 'KEEP', 'Exposure-normalized claims frequency.'],
    ['latest_claim_severity', 'B3', 'Mean paid amount per FY2025 episode', 'Yes', 'Low', 'KEEP', 'Separates severity from frequency.'],
    ['latest_claim_count', 'B3', 'FY2025 paid episode count', 'Yes', 'Low', 'KEEP', 'Claims volume with exposure denominator.'],
    ['covered_lives_2025', 'B1', 'FY2025 covered lives', 'Yes', 'Low', 'KEEP', 'Account size and exposure.'],
    ['latest_risk_score', 'B1', 'FY2025 account risk score', 'Yes', 'Low', 'KEEP', 'Current risk signal.'],
    ['risk_band', 'B1', 'Risk-score band', 'Yes', 'Low', 'INVESTIGATE', 'Interpretable alternative to continuous score; avoid duplication.'],
    ['rating_region', 'B1', 'Account rating region', 'Yes', 'Low', 'KEEP', 'Regional pricing and market context.'],
    ['unemployment_rate_2025', 'B4', '2025 regional unemployment rate', 'Yes', 'Low', 'INVESTIGATE', 'Relevant regional context; validate incremental value later.'],
    ['renewal_rate_3yr', 'B2', 'Mean renewed_flag over 2023-2025', 'Conditional', 'Medium', 'KEEP', 'Historical retention signal; time-align for validation.'],
    ['rate_change_mean_3yr', 'B2', 'Mean offered rate change over 2023-2025', 'Yes', 'Low', 'KEEP', 'Historical price exposure.'],
    ['rate_change_slope', 'B2', '2025 minus 2023 offered rate change divided by 2', 'Yes', 'Low', 'INVESTIGATE', 'Historical price trajectory from only three observations.'],
    ['risk_score_change', 'B2', '2025 quote risk score minus 2023 quote risk score', 'Yes', 'Low', 'INVESTIGATE', 'Historical risk direction.'],
    ['historical_loss_making_indicator', 'B2+B3', 'FY2025 MLR > 1', 'Yes', 'Low', 'KEEP', 'Simple profitability indicator.'],
    ['three_year_average_mlr', 'B3', 'Average MLR for 2023-2025', 'No', 'High', 'DROP', 'B3 has FY2025 claims only; do not fabricate history.'],
    ['claims_trend', 'B3', 'Annual claims trend', 'No', 'High', 'DROP', 'B3 has FY2025 claims only.'],
    ['medical_cost_inflation_pct', 'B4', 'Medical inflation benchmark', 'No for primary case', 'High', 'DROP', 'Primary case prohibits added medical inflation.'],
    ['utilization_trend_pct', 'B4', 'Utilization trend benchmark', 'No for primary case', 'High', 'DROP', 'Primary case prohibits added utilization trend.'],
    ['renewed_2026', 'Future', '2026 renewal outcome', 'No', 'Direct', 'DROP', 'Future target/post-renewal leakage.'],
    ['account_id', 'B1', 'Account identifier', 'Yes', 'Memorization', 'DROP', 'Join/output key only, not a predictor.'],
    ['account_status', 'B1', 'Current account status', 'Yes', 'Low', 'DROP', 'Constant Active value.'],
], columns=['feature', 'source', 'definition', 'available_at_2026_decision', 'leakage_risk', 'decision', 'reason'])
decisions.to_csv(OUT / 'feature_decisions.csv', index=False); display(decisions)

## Candidate 2026 features entering model validation

The final feature list is a clean candidate universe only. No model coefficients or optimization results are used for selection.

In [ ]:
keep = decisions.loc[decisions.decision.eq('KEEP'), 'feature'].tolist()
drop = decisions.loc[decisions.decision.eq('DROP'), 'feature'].tolist()
leakage_issues = 'None in the corrected historical renewal analysis. FY2025 claims/MLR are restricted to 2026 profitability context; quote-year risk and matching-year B4 regional values are used for 2023-2025 response analysis.'
loss_share = a25.loss_making.mean()
high_risk_claim_share = a25.loc[a25.risk_score_2025.ge(a25.risk_score_2025.median()), 'paid_claims'].sum() / a25.paid_claims.sum()
portfolio_price_response = renewal_by_rate_year.groupby('quote_year').apply(lambda d: d.loc[d.renewal_probability.notna(), 'renewal_probability'].iloc[0] - d.loc[d.renewal_probability.notna(), 'renewal_probability'].iloc[-1], include_groups=False).to_dict()
summary = f'''# EDA summary: Track A 2026 renewal pricing

## Profitability

FY2025 mean MLR is {a25.mlr.mean():.1%}; median MLR is {a25.mlr.median():.1%}; {loss_share:.1%} of accounts are loss-making. Accounts at or above the median FY2025 risk score generate {high_risk_claim_share:.1%} of paid claims.

Finding -> FY2025 profitability is heterogeneous and concentrated in higher-risk accounts. Business implication -> use account-level claims and risk information for 2026 pricing. Modelling implication -> retain latest MLR, frequency, severity, exposure and current risk candidates.

## Renewal and price response

For each quote year, renewal analysis uses only `risk_score_at_quote`, `quoted_rate_change_pct`, `renewed_flag`, stable B1 account attributes, and matching-year B4 regional values. The observed renewal drop from the lowest to highest populated rate band is approximately {', '.join(f'{year}: {value:.0%}' for year, value in portfolio_price_response.items())}. The four-panel analysis compares quote-time risk band, industry, account size and quote-year regional economic tier with sample counts.

Finding -> Price increases have an observable retention trade-off and curves differ across segments. Business implication -> broad increases can create adverse selection. Modelling implication -> preserve quote-year response evidence and interactions for later validation; do not interpret descriptive curves causally.

Retrospective-only note -> FY2025 MLR/loss-making is used for the 2026 profitability analysis only. It is not used to explain 2023 or 2024 renewal behaviour.

## Temporal stability

B2 renewal rate, offered rate change, quoted premium and `risk_score_at_quote` are compared using each quote year's data. Matching-year B4 benchmarks are used for regional response analysis. B3 is FY2025-only, so claims/MLR/frequency/severity stability across years cannot be assessed.

Finding -> B2 supports historical validation, but claims trends need additional years. Business implication -> do not claim a multi-year claims trend from this release. Modelling implication -> use FY2025 claims as latest experience only.

## Regional context

The regional response analysis uses B4 benchmarks matched to each B2 quote year. B4 medical inflation and utilization trend are excluded from the primary case because FY2025 paid claims are carried into 2026 without either uplift.

## Redundancy and leakage

MLR is deterministically paid claims divided by premium; `risk_band` is derived from the current B1 risk score and is reserved for 2026 interpretation, while historical renewal uses quote-year `risk_score_at_quote`; IDs are excluded as predictors; future outcomes and post-renewal information are excluded. No coefficients were used for selection.

## Candidate 2026 features entering model validation

{', '.join(keep)}

## Dropped features

{', '.join(drop)}

No ML model or optimization was run.'''
(OUT / 'EDA_summary.md').write_text(summary, encoding='utf-8')
print(summary)
print('EDA complete:', 'YES')
print('Remaining leakage issues:', leakage_issues)
print('Final candidate 2026 feature list:', ', '.join(keep))